In [1]:
# Data manipulation
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from itertools import combinations


# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Statistics and modeling
from scipy.stats import gaussian_kde
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

# Preprocessing

In [14]:
data = pd.read_csv('/Users/go202/Desktop/equate_sheet/final_df.csv')
data.loc[data['language_name'] == 'Mandarin Chinese', 'models_n'] = 11443
data.loc[data['iso_code'] == 'unk', 'models_n'] = 2
# Identify the rows for China
china_rows = data[data['country_iso_code'] == 'CHN']

# Define columns to exclude
excluded_columns = ['avg_latency_ms', 'distance_to_closest_university_km', 'distance_to_second_closest_university_km']

# Get numeric columns in the data
numeric_columns = data.select_dtypes(include=[np.number]).columns

# Filter out the excluded columns
columns_to_update = [col for col in numeric_columns if col not in excluded_columns]

# Compute max values for those columns across China rows
china_max_values = china_rows[columns_to_update].max()

# Mask for Mandarin Chinese
mandarin_mask = data['language_name'] == 'Mandarin Chinese'

# Replace values in Mandarin Chinese row with China max values (only for selected columns)
for col in columns_to_update:
    data.loc[mandarin_mask, col] = china_max_values[col]
def cc_to_numeric(value):
    if isinstance(value, (int, float)):
        return value

    value = value.strip()
    if 'K' in value:
        return float(value.replace('K', '').strip()) * 1e3
    elif 'M' in value:
        return float(value.replace('M', '').strip()) * 1e6
    elif 'G' in value:
        return float(value.replace('G', '').strip()) * 1e9
    elif 'T' in value:
        return float(value.replace('T', '').strip()) * 1e12
    else:
        # Handle plain numeric strings
        try:
            return float(value)
        except ValueError:
            return np.nan  # or 0, depending on how you want to handle junk data

# Apply to your dataframe(s)
data['cc_gb'] = data['cc_gb'].apply(cc_to_numeric)

def wiki_to_numeric(value):
    if isinstance(value, int):
        return value
    if pd.isna(value):
        return 0  
    return int(value.replace(',', ''))


data['wiki_articles_n'] = data['wiki_articles_n'].apply(wiki_to_numeric)

# Weights and Feature Groups Specifications

In [19]:
binary_features = ['bible_exists', 'cybersecurity_law_exists','institutional_status']
log_features = [    'cc_gb',
    'models_n',              
    'opus_gb',
    'archival_gb',
    'xeus_gb', 
    'wikitongues_gb', 'n_speakers', 
    'wiki_active_users'
    ]

In [15]:
def compute_normalized_weights(features, ranks, group_name=None):
    """
    Computes normalized linear weights from ranks.

    Args:
        features (list): List of feature names.
        ranks (list): Corresponding rank values.
        group_name (str, optional): Name of group for display/debug.

    Returns:
        pd.DataFrame: DataFrame with 'feature', 'rank', 'linear_weight', 'normalized_weight'
    """
    n = len(ranks)
    linear_weights = [1 - (rank - 1) / (n - 1) for rank in ranks]
    total = sum(linear_weights)
    normalized_weights = [w / total for w in linear_weights]

    df = pd.DataFrame({
        "feature": features,
        "rank": ranks,
        "linear_weight": linear_weights,
        "normalized_weight": normalized_weights
    })

    if group_name:
        df["group"] = group_name  # Optional for tracking

    return df


# Define groups and ranks in a dictionary
feature_groups = {
    "technical": {
        "features": ['cc_gb', 'models_n', 'opus_gb', 'archival_gb', 'bible_exists', 'xeus_gb', 'extra_feature'],  # example 7 features
        "ranks": [1, 2, 3, 4, 5, 6, 7]
    },
    "internet": {
        "features": ['households_with_phones_pct', 'individuals_using_internet_pct',
                     'avg_download_speed_kbps', 'households_with_home_internet_pct',
                     'cybersecurity_law_exists'],
        "ranks": [1, 2, 3, 4, 5]
    },
    "language": {
        "features": ['institutional_status', 'n_speakers', 'wiki_active_users', 'literacy_rate',
                     'admin_1_gdp', 'rnd_gdp_pct', 'admin_1_hdi', 'admin_1_education_level'],
        "ranks": [1, 2, 3, 4, 5, 6, 7, 8]
    }
}

feature_group_definitions = {
    group: values["features"]
    for group, values in feature_groups.items()
}
# Compute weights for each group
all_feature_weights = []
for group_name, group_data in feature_groups.items():
    df_weights = compute_normalized_weights(
        features=group_data["features"],
        ranks=group_data["ranks"],
        group_name=group_name
    )
    all_feature_weights.append(df_weights)

# Concatenate into final DataFrame
final_feature_weights_df = pd.concat(all_feature_weights, ignore_index=True)

# Display result
print(final_feature_weights_df)

                              feature  rank  linear_weight  normalized_weight  \
0                               cc_gb     1       1.000000           0.285714   
1                            models_n     2       0.833333           0.238095   
2                             opus_gb     3       0.666667           0.190476   
3                         archival_gb     4       0.500000           0.142857   
4                        bible_exists     5       0.333333           0.095238   
5                             xeus_gb     6       0.166667           0.047619   
6                       extra_feature     7       0.000000           0.000000   
7          households_with_phones_pct     1       1.000000           0.400000   
8      individuals_using_internet_pct     2       0.750000           0.300000   
9             avg_download_speed_kbps     3       0.500000           0.200000   
10  households_with_home_internet_pct     4       0.250000           0.100000   
11           cybersecurity_l

In [20]:
def merge_correlated_features(df, feature_group_definitions, final_feature_weights_df, threshold=0.85):
    merged_df = df.copy()
    updated_group_defs = {}
    updated_weights = final_feature_weights_df.copy()

    for group_name, features in feature_group_definitions.items():
        valid_features = [f for f in features if f in df.columns]
        if len(valid_features) < 2:
            updated_group_defs[group_name] = valid_features
            continue

        corr_matrix = df[valid_features].corr(method='pearson').abs()
        merged = set()
        merged_features = []

        for f1, f2 in combinations(valid_features, 2):
            if corr_matrix.loc[f1, f2] > threshold:
                if f1 not in merged and f2 not in merged:
                    w1 = final_feature_weights_df.loc[final_feature_weights_df['feature'] == f1, 'normalized_weight'].values[0]
                    w2 = final_feature_weights_df.loc[final_feature_weights_df['feature'] == f2, 'normalized_weight'].values[0]
                    merged_name = f"{f1}_{f2}_merged"
                    merged_df[merged_name] = (df[f1]*w1 + df[f2]*w2) / (w1 + w2)

                    merged_features.append(merged_name)
                    updated_weights = updated_weights[~updated_weights['feature'].isin([f1, f2])]
                    updated_weights = pd.concat([
                        updated_weights,
                        pd.DataFrame({'feature': [merged_name], 'normalized_weight': [w1 + w2]})
                    ])
                    merged.update([f1, f2])

        unmerged_features = [f for f in valid_features if f not in merged]
        updated_group_defs[group_name] = unmerged_features + merged_features

    return merged_df, updated_group_defs, updated_weights

In [21]:
merged_data, updated_group_defs, updated_weights = merge_correlated_features(
    df=data,
    feature_group_definitions=feature_group_definitions,
    final_feature_weights_df=final_feature_weights_df,
    threshold=0.85  # Adjust this threshold if needed
)

In [22]:
updated_group_defs

{'technical': ['opus_gb',
  'archival_gb',
  'bible_exists',
  'xeus_gb',
  'cc_gb_models_n_merged'],
 'internet': ['avg_download_speed_kbps',
  'households_with_home_internet_pct',
  'cybersecurity_law_exists',
  'households_with_phones_pct_individuals_using_internet_pct_merged'],
 'language': ['institutional_status',
  'n_speakers',
  'wiki_active_users',
  'literacy_rate',
  'admin_1_gdp',
  'rnd_gdp_pct',
  'admin_1_hdi_admin_1_education_level_merged']}

# Compute full index 

In [ ]:
def adjusted_geometric_weighted_aggregation_individual_weights(
    df,
    final_feature_weights_df,
    feature_group_definitions,
    binary_features,
    log_scale_features=None,
    language_id_column='language_name', 
    epsilon=1e-6
):
    """
    Calculates an aggregated readiness index using weighted geometric aggregation
    with adjustments for correlated features, log scaling, and binary penalties.

    Args:
        df (pd.DataFrame): The input DataFrame containing feature values for languages.
        final_feature_weights_df (pd.DataFrame): DataFrame with 'feature' and 'normalized_weight' columns.
        feature_group_definitions (dict): A dictionary mapping group names to lists of features in that group.
        binary_features (list): A list of feature names that are binary (0 or 1).
        log_scale_features (list, optional): A list of continuous feature names to apply log scaling to. Defaults to None.
        language_id_column (str, optional): The name of the column containing language identifiers. Not directly used in calculation.
        epsilon (float, optional): Small constant added before log-scaling to prevent log(0). Defaults to 1e-6.

    Returns:
        pd.Series: A pandas Series of aggregated readiness scores, sorted in descending order.
    """

    # Merge correlated features (Assuming this modifies df, feature_group_definitions, weights_df in place or returns new ones)
    df, feature_group_definitions, final_feature_weights_df = merge_correlated_features(
        df, feature_group_definitions, final_feature_weights_df
    )

    # Replace zero weights with small positive value (if they exist)
    final_feature_weights_df['normalized_weight'] = final_feature_weights_df['normalized_weight'].replace(0, 0.01)
    feature_weight_map = final_feature_weights_df.set_index('feature')['normalized_weight'].to_dict()

    num_groups = len(feature_group_definitions)
    df_processed = df.copy()

    # Identify all non-binary numeric features that will be processed
    non_binary_numeric_features = []
    for feature in df_processed.columns:
        if feature not in binary_features and pd.api.types.is_numeric_dtype(df_processed[feature]):
            non_binary_numeric_features.append(feature)

    # Shifting Continuous Features: Add 1 to all non-binary numeric features to avoid zero values
    for feature in non_binary_numeric_features:
        df_processed[feature] = df_processed[feature].fillna(0) + 1 

    # Log-Transform (Optional): Apply log scaling to selected continuous features
    if log_scale_features:
        for feature in log_scale_features:
            if feature in df_processed.columns and feature in non_binary_numeric_features: # Ensure it's a non-binary numeric feature
                df_processed[feature] = np.log(df_processed[feature] + epsilon)
            else:
                print(f"Warning: Log scale feature '{feature}' not found or is binary/non-numeric. Skipping log transform.")

    # Normalize all non-binary features to a [0, 1] range (Min-Max Scaling)
    for feature in non_binary_numeric_features:
        min_val = df_processed[feature].min()
        max_val = df_processed[feature].max()

        if max_val == min_val: # Avoid division by zero if all values are the same
            df_processed[feature] = 0.5 # Assign a neutral value, or 1 if presence is always positive
        else:
            df_processed[feature] = (df_processed[feature] - min_val) / (max_val - min_val)
            df_processed[feature] = df_processed[feature].apply(lambda x: max(x, epsilon))
   

    aggregated_values = []

    for idx, row in df_processed.iterrows():
        group_multiplicative_factor = 1.0

        for group_name, features_in_group in feature_group_definitions.items():
            group_geometric_product_sum_weighted = 0.0 # Will accumulate log-transformed weighted sum
            group_weight_sum = 0.0
            valid_features_in_group = 0

            # Calculate sum of log(val) * weight for the group
            for feature in features_in_group:
                if feature in df_processed.columns and feature in feature_weight_map and feature not in binary_features:
                    val = row[feature]
                    weight = feature_weight_map[feature]
                    group_geometric_product_sum_weighted += np.log(val) * weight
                    group_weight_sum += weight
                    valid_features_in_group += 1

            if valid_features_in_group > 0:
                if abs(group_weight_sum - 1.0) > 1e-6:
                    # Renormalize weights and apply geometric mean: exp(sum(renormalized_weight * log(val)))
                    renormalized_log_sum = 0.0
                    for feature in features_in_group:
                        if feature in df_processed.columns and feature in feature_weight_map and feature not in binary_features:
                            original_weight = feature_weight_map[feature]
                            renormalized_weight = original_weight / group_weight_sum
                            renormalized_log_sum += np.log(row[feature]) * renormalized_weight
                    group_multiplicative_factor *= np.exp(renormalized_log_sum)
                else:
                    # If weights already sum to 1, apply geometric mean directly
                    group_multiplicative_factor *= np.exp(group_geometric_product_sum_weighted)
            # If no valid features in group, group_multiplicative_factor remains 1.0, effectively no contribution.

        # Binary Feature Adjustment
        binary_penalty = 1.0
        for binary_feature in binary_features:
            if binary_feature in df_processed.columns and binary_feature in feature_weight_map:
                weight = feature_weight_map[binary_feature]
                val = row[binary_feature]

                penalty_factor = 1 - weight * (1 - val)
                # Ensure penalty factor is not negative (if weight > 1) or leads to division by zero later
                binary_penalty *= max(0, penalty_factor) # Ensure non-negative penalty
            else:
                print(f"Warning: Binary feature '{binary_feature}' not found in data or weights. Assuming no penalty.")

        # Final Index Score
    
        final_score = (group_multiplicative_factor ** (1 / num_groups)) * binary_penalty
        aggregated_values.append(final_score)

    return pd.Series(aggregated_values, index=df.index).sort_values(ascending=False)

In [ ]:
aggregated_result = adjusted_geometric_weighted_aggregation_individual_weights(
    df=merged_data,
    final_feature_weights_df=updated_weights,
    feature_group_definitions=updated_group_defs,
    binary_features=binary_features,
    log_scale_features=log_features,
    language_id_column='language_name'  
)

print(aggregated_result)

1705    0.748709
1611    0.499661
2755    0.372975
937     0.369301
2728    0.345428
          ...   
5196    0.000019
5191    0.000017
5195    0.000009
5194    0.000008
5193    0.000008
Length: 6003, dtype: float64


# Genereal Index Frame

In [26]:
aggregated_df = aggregated_result.rename("aggregated_score").to_frame()

# Merge on the index
complete_index = aggregated_df.merge(data, left_index=True, right_index=True, how='inner')

# Compute group wise Index

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations

# --- Helper Functions for Global Preprocessing ---

def _merge_correlated_features_globally(df, feature_group_definitions, feature_weights_df, binary_features, correlation_threshold=0.85):
    """
    Identifies and prepares for merging highly correlated continuous features globally.
    This function modifies the feature_group_definitions and feature_weights_df in place
    to reflect the merged features, and returns instructions for df modification.

    Args:
        df (pd.DataFrame): The original DataFrame.
        feature_group_definitions (dict): Dict mapping group names to lists of features. (Modified in place)
        feature_weights_df (pd.DataFrame): DataFrame of feature weights. (Modified in place)
        binary_features (list): List of binary feature names.
        correlation_threshold (float): Pearson correlation threshold for merging.

    Returns:
        list: A list of dicts, each specifying how to create a new merged column.
    """
    
    # Work with copies to avoid modifying original objects prematurely
    current_feature_group_defs = {k: list(v) for k, v in feature_group_definitions.items()}
    current_weights = feature_weights_df.copy()
    current_weight_map = current_weights.set_index('feature')['normalized_weight'].to_dict()

    all_numeric_features_for_correlation = []
    # Collect all non-binary numeric features across all groups for correlation check
    for group_feats in current_feature_group_defs.values():
        for feat in group_feats:
            if feat in df.columns and pd.api.types.is_numeric_dtype(df[feat]) and feat not in binary_features:
                all_numeric_features_for_correlation.append(feat)
    all_numeric_features_for_correlation = list(set(all_numeric_features_for_correlation)) # Remove duplicates

    if not all_numeric_features_for_correlation:
        print("Warning: No numeric non-binary features found for correlation analysis. Skipping merging.")
        return [], current_feature_group_defs, current_weights # No merges to perform

    corr_matrix_global = df[all_numeric_features_for_correlation].corr().abs()
    
    features_to_merge_pairs = []
    processed_features_for_selection = set() # Track features already picked for a merge pair

    for f1, f2 in combinations(all_numeric_features_for_correlation, 2):
        if corr_matrix_global.loc[f1, f2] > correlation_threshold:
            if f1 not in processed_features_for_selection and f2 not in processed_features_for_selection:
                features_to_merge_pairs.append((f1, f2))
                processed_features_for_selection.add(f1)
                processed_features_for_selection.add(f2)

    new_merged_columns_info = [] # Stores info needed to create columns later
    
    for f1, f2 in features_to_merge_pairs:
        w1 = current_weight_map.get(f1, 0)
        w2 = current_weight_map.get(f2, 0)

        # Handle cases where weights might be zero (avoid division by zero)
        if w1 + w2 == 0:
            print(f"Warning: Cannot merge {f1} and {f2} due to zero total weight. Skipping this merge.")
            continue

        new_feat_name = f"{f1}_{f2}_merged"
        new_merged_columns_info.append({
            'name': new_feat_name,
            'original_features': (f1, f2),
            'original_weights': (w1, w2)
        })

        # Update feature_weights_df
        current_weights = current_weights[~current_weights['feature'].isin([f1, f2])]
        current_weights = pd.concat([current_weights,
                                     pd.DataFrame({'feature': [new_feat_name], 'normalized_weight': [w1 + w2]})],
                                    ignore_index=True)
        current_weight_map[new_feat_name] = w1 + w2 # Update map for subsequent lookups

        # Update feature_group_definitions for each group
        for group_name, features_in_group in current_feature_group_defs.items():
            if f1 in features_in_group or f2 in features_in_group:
                updated_list = [new_feat_name if (item == f1 or item == f2) else item for item in features_in_group]
                current_feature_group_defs[group_name] = list(set(updated_list)) # Use set to remove duplicates if both original features were in the same group

    return new_merged_columns_info, current_feature_group_defs, current_weights

def _add_one_to_continuous_features(df, features_list):
    """Adds 1 to specified continuous numeric features."""
    for feature in features_list:
        if feature in df.columns:
            # Fill NaNs with 0 before adding 1 to ensure numeric type and handle missing data
            df[feature] = pd.to_numeric(df[feature].fillna(0)) + 1
    return df

def _apply_log_scale_globally(df, features_list, epsilon):
    """Applies log scaling to specified features."""
    for feature in features_list:
        if feature in df.columns:
            df[feature] = np.log(df[feature] + epsilon)
        else:
            print(f"Warning: Log scale feature '{feature}' not found in DataFrame. Skipping log transform.")
    return df

def _normalize_continuous_features_globally(df, features_list, epsilon):
    """Applies Min-Max normalization to [epsilon, 1] for specified features."""
    for feature in features_list:
        if feature in df.columns:
            min_val = df[feature].min()
            max_val = df[feature].max()

            if max_val == min_val:
                # If all values are the same, assign a neutral value (e.g., 0.5)
                # This prevents division by zero and ensures a non-zero value for geometric mean.
                df[feature] = 0.5
            else:
                df[feature] = (df[feature] - min_val) / (max_val - min_val)
                # Ensure values are strictly positive for geometric mean (min value is epsilon)
                df[feature] = df[feature].apply(lambda x: max(x, epsilon))
    return df

def _calculate_weighted_geometric_mean_series(df_processed, features_in_group, feature_weight_map, binary_features_global, epsilon):
    """
    Calculates the weighted geometric mean for a group's features across all languages (Series).
    Applies binary penalties for features within this group.
    """
    
    # Initialize a Series for the sum of log-weighted values
    log_sum_weighted = pd.Series(0.0, index=df_processed.index)
    total_weight_for_geometric_mean = 0.0

    # Separate continuous and binary features within this group
    group_continuous_features = [f for f in features_in_group if f not in binary_features_global]
    group_binary_features = [f for f in features_in_group if f in binary_features_global]

    # Calculate the weighted sum of logs for continuous features
    for feature in group_continuous_features:
        if feature in df_processed.columns and feature in feature_weight_map:
            weight = feature_weight_map[feature]
            if weight > 0: # Only include features with positive weights in the sum
                log_sum_weighted += np.log(df_processed[feature]) * weight
                total_weight_for_geometric_mean += weight
        else:
            print(f"Warning: Group feature '{feature}' not found in processed data or weights. Skipping for geometric mean.")

    # Compute the geometric mean component
    geometric_mean_component = pd.Series(1.0, index=df_processed.index)
    if total_weight_for_geometric_mean > 0:
        # Renormalize weights implicitly by dividing the sum of logs by total weight, then exponentiate
        geometric_mean_component = np.exp(log_sum_weighted / total_weight_for_geometric_mean)
    else:
        # If no valid continuous features, it has a neutral effect (1.0)
        pass # geometric_mean_component already 1.0

    # Apply binary feature penalties relevant to this group
    binary_penalty_component = pd.Series(1.0, index=df_processed.index)
    for b_feature in group_binary_features:
        if b_feature in df_processed.columns and b_feature in feature_weight_map:
            b_weight = feature_weight_map[b_feature]
            b_val = df_processed[b_feature]
            
            # Ensure weight is within [0,1] for proper penalty behavior
            # This is crucial for the penalty to behave as a reduction [0,1].
            clipped_b_weight = max(0.0, min(1.0, b_weight))
            
            # Penalty form: 1 - w_b * (1 - x_i,b)
            penalty_factor_series = 1 - clipped_b_weight * (1 - b_val)
            binary_penalty_component *= np.maximum(0, penalty_factor_series) # Ensure non-negative penalty

        else:
            print(f"Warning: Binary feature '{b_feature}' for group not found in processed data or weights. Assuming no penalty.")

    return geometric_mean_component * binary_penalty_component

def compute_group_indices(
    df,
    final_feature_weights_df,
    feature_group_definitions,
    binary_features,
    log_scale_features=None,
    epsilon=1e-6,
    correlation_threshold=0.85
):
    """
    Calculates indices for each feature group for all languages,
    incorporating merging of correlated features, shifting, log-scaling,
    normalization, weighted geometric mean aggregation, and binary penalties.

    Args:
        df (pd.DataFrame): The input DataFrame containing feature values for languages.
        final_feature_weights_df (pd.DataFrame): DataFrame with 'feature' and 'normalized_weight' columns.
                                                (These weights are for individual features, not group weights).
        feature_group_definitions (dict): A dictionary mapping group names to lists of features in that group.
        binary_features (list): A list of global binary feature names (0 or 1).
        log_scale_features (list, optional): A list of continuous feature names to apply log scaling to. Defaults to None.
        epsilon (float, optional): Small constant added before log-scaling and after normalization
                                    to prevent log(0) or zero values in geometric mean. Defaults to 1e-6.
        correlation_threshold (float, optional): Pearson correlation threshold for merging features. Defaults to 0.85.

    Returns:
        pd.DataFrame: A DataFrame where each column represents a feature group index,
                      and rows correspond to languages. An 'Overall_Readiness_Score' is also included.
    """
    df_processed = df.copy()
    
    # Ensure all initial weights are positive for robustness in merges/calculations
    # Replace 0 weights with a small value to prevent division by zero or errors later.
    final_feature_weights_df['normalized_weight'] = final_feature_weights_df['normalized_weight'].replace(0, 0.01)

    # Make copies of definitions and weights to modify them for merging
    current_feature_groups_defs = {k: list(v) for k, v in feature_group_definitions.items()}
    current_weights_df = final_feature_weights_df.copy()

    # 1. Global Merging of Correlated Features
    # This identifies features to merge, updates `current_feature_groups_defs`
    # and `current_weights_df` and returns instructions for creating new columns.
    merged_features_info, current_feature_groups_defs, current_weights_df = \
        _merge_correlated_features_globally(
            df_processed, current_feature_groups_defs, current_weights_df, binary_features, correlation_threshold
        )
    
    # Create the new merged feature columns in df_processed
    features_to_drop_after_merge = set()
    for merge_info in merged_features_info:
        f1, f2 = merge_info['original_features']
        w1, w2 = merge_info['original_weights']
        new_feat_name = merge_info['name']
        
        # Calculate the merged feature value using arithmetic mean as per your framework
        # Apply fillna(0) for safety consistent with how other features are handled for +1 shift
        df_processed[new_feat_name] = (df_processed[f1].fillna(0) * w1 + df_processed[f2].fillna(0) * w2) / (w1 + w2)
        
        features_to_drop_after_merge.add(f1)
        features_to_drop_after_merge.add(f2)

    # Drop the original merged features that are now replaced by composite ones
    df_processed = df_processed.drop(columns=list(features_to_drop_after_merge), errors='ignore')

    # Create a unified weight map for all features (original + newly merged)
    final_weight_map = current_weights_df.set_index('feature')['normalized_weight'].to_dict()

    # Identify all non-binary numeric features that are *currently in the DataFrame*
    # after potential merging, to apply global preprocessing steps
    all_active_non_binary_numeric_features = [
        f for f in df_processed.columns
        if f not in binary_features and pd.api.types.is_numeric_dtype(df_processed[f])
    ]

    # 2. Shifting Continuous Features (Add 1)
    df_processed = _add_one_to_continuous_features(df_processed, all_active_non_binary_numeric_features)

    # 3. Log-Transform (Optional)
    df_processed = _apply_log_scale_globally(df_processed, log_scale_features or [], epsilon)

    # 4. Normalize Continuous Features
    df_processed = _normalize_continuous_features_globally(df_processed, all_active_non_binary_numeric_features, epsilon)

    # --- Calculate Group Indices ---
    group_indices_df = pd.DataFrame(index=df.index)

    for group_name, features_in_group_merged in current_feature_groups_defs.items():
        # Calculate index for each language within the current group
        group_index_series = _calculate_weighted_geometric_mean_series(
            df_processed,
            features_in_group_merged, # This list contains potentially merged features
            final_weight_map,
            binary_features, # Global list of binary features to check for per-group application
            epsilon
        )
        group_indices_df[group_name] = group_index_series

    # --- Calculate Overall Readiness Score (as per Section 7 & 9 of mathematical framework) ---

    # S_i_groups = (product(G_i,g))^(1/G)
    # This calculates the unweighted geometric mean of all group indices
    product_of_group_indices = group_indices_df.prod(axis=1)
    num_groups = len(current_feature_groups_defs)
    S_i_groups = product_of_group_indices ** (1 / num_groups) if num_groups > 0 else pd.Series(1.0, index=df.index)

    # P_i_binary (Global Binary Penalty - Section 8)
    # This is applied to all global binary features not tied to specific groups for penalty calculation
    P_i_binary_global = pd.Series(1.0, index=df.index)
    

    # Recalculate group_indices_df without binary penalties
    group_indices_df_continuous_only = pd.DataFrame(index=df.index)
    for group_name, features_in_group_merged in current_feature_groups_defs.items():
        # _calculate_weighted_geometric_mean_series needs to be updated to NOT include binary penalties
        # For simplicity, let's just extract continuous features in this call
        group_continuous_features = [f for f in features_in_group_merged if f not in binary_features]
        
        # Calculate geometric mean of continuous features for this group
        group_geometric_mean_log_sum = pd.Series(0.0, index=df_processed.index)
        total_weight_for_geometric_mean = 0.0

        for feature in group_continuous_features:
            if feature in df_processed.columns and feature in final_weight_map:
                weight = final_weight_map[feature]
                if weight > 0:
                    group_geometric_mean_log_sum += np.log(df_processed[feature]) * weight
                    total_weight_for_geometric_mean += weight

        group_score_continuous = pd.Series(1.0, index=df_processed.index)
        if total_weight_for_geometric_mean > 0:
            group_score_continuous = np.exp(group_geometric_mean_log_sum / total_weight_for_geometric_mean)
        
        group_indices_df_continuous_only[group_name] = group_score_continuous


    # S_i_groups (unweighted geometric mean of continuous-only group scores)
    product_of_group_indices_continuous = group_indices_df_continuous_only.prod(axis=1)
    S_i_groups_final = product_of_group_indices_continuous ** (1 / num_groups) if num_groups > 0 else pd.Series(1.0, index=df.index)

    # 8. Global Binary Feature Adjustment (P_i_binary)
    P_i_binary_global = pd.Series(1.0, index=df.index)
    for binary_feature in binary_features:
        if binary_feature in df_processed.columns and binary_feature in final_weight_map:
            weight = final_weight_map[binary_feature]
            val = df_processed[binary_feature] # 0 or 1
            clipped_weight = max(0.0, min(1.0, weight)) # Ensure 0 <= weight <= 1
            penalty_factor_series = 1 - clipped_weight * (1 - val)
            P_i_binary_global *= np.maximum(0, penalty_factor_series)
        else:
            print(f"Warning: Global binary feature '{binary_feature}' not found in processed data or weights. Assuming no penalty.")

    # 9. Final Index Score (Overall readiness score)
    overall_readiness_scores = S_i_groups_final * P_i_binary_global

    # The request was to calculate indices also for feature groups: one index for language, one index for technical etc.
    # This implies returning the individual group indices (continuous-only) AND the final overall index.
    
    # Add the overall score to the DataFrame that contains individual group scores
    group_indices_df_continuous_only['Overall_Readiness_Score'] = overall_readiness_scores

    return group_indices_df_continuous_only.sort_values(by='Overall_Readiness_Score', ascending=False)

In [28]:
group_indices = compute_group_indices(
    data,
    final_feature_weights_df,
    feature_group_definitions,
    binary_features,
    log_scale_features = log_features,
    epsilon=1e-6,
    correlation_threshold=0.85)

# Full Index Frame

In [30]:
overall_and_groupwise_index = complete_index.join(group_indices, how='left')

In [ ]:
overall_and_groupwise_index.to_csv('complete_index.csv')